# Phase 5 v8b Notebook
No file IO in cells. Inline checks only.

In [ ]:
import math, itertools
import matplotlib.pyplot as plt
def units_mod(m): return [u for u in range(m) if math.gcd(u,m)==1]
def cyclic_key(N,t):
    mod=2*N
    return min((t*u*u)%mod for u in units_mod(N))
cases=[(N,t) for N in range(2,25) for t in range(1,2*N) if math.gcd(t,N)==1]
checks=[]
for N,t in cases:
    us=units_mod(N); u=us[(N+t)%len(us)]
    checks.append(cyclic_key(N,t)==cyclic_key(N,(t*u*u)%(2*N)))
plt.figure(); plt.bar(['pass','fail'], [sum(checks), len(checks)-sum(checks)]); plt.title('Cyclic unit-transform invariance')
print('PASS' if all(checks) else 'FAIL', {'cases':len(checks),'failures':len(checks)-sum(checks)})


In [ ]:
def legendre(a,p):
    a%=p
    if a==0: return 0
    r=pow(a,(p-1)//2,p)
    return -1 if r==p-1 else r
def gl2(p):
    return [((a,b),(c,d)) for a,b,c,d in itertools.product(range(p), repeat=4) if (a*d-b*c)%p!=0]
def trans(A,P,p):
    a,b=A[0]; c=A[1][1]; p00,p01=P[0]; p10,p11=P[1]
    AP00=(a*p00+b*p10)%p; AP01=(a*p01+b*p11)%p; AP10=(b*p00+c*p10)%p; AP11=(b*p01+c*p11)%p
    B00=(p00*AP00+p10*AP10)%p; B01=(p00*AP01+p10*AP11)%p; B11=(p01*AP01+p11*AP11)%p
    return ((B00,B01),(B01,B11))
summary=[]
for p in [3,5,7]:
    G=gl2(p); classes={}
    for a,b,c in itertools.product(range(p), repeat=3):
        det=(a*c-b*b)%p
        if det==0: continue
        A=((a,b),(b,c)); can=min(tuple(x for row in trans(A,P,p) for x in row) for P in G)
        classes.setdefault(legendre(det,p), set()).add(can)
    summary.append((p,{k:len(v) for k,v in classes.items()}))
plt.figure(); plt.plot([p for p,_ in summary], [sum(d.values()) for _,d in summary], marker='o'); plt.title('Odd field rank-2 isometry classes')
ok=all(all(v==1 for v in d.values()) and len(d)==2 for _,d in summary)
print('PASS' if ok else 'FAIL', {'summary':summary})


In [ ]:
blocks=[(3,1),(4,1),(5,1),(8,3),(9,1),(16,5)]
def sym(blks): return '|'.join(sorted(f'{N}:{cyclic_key(N,t)}' for N,t in blks))
base=sym(blocks); rev=sym(list(reversed(blocks)))
plt.figure(); plt.bar(['base_hash','rev_hash'], [hash(base)%1000, hash(rev)%1000]); plt.title('Direct-sum symbol permutation check')
print('PASS' if base==rev else 'FAIL', {'rank_blocks':len(blocks),'symbol_equal':base==rev})


In [ ]:
rows=[]
for rank in [8,16,32,64]:
    enum=False; orbit=False
    rows.append((rank, enum, orbit))
plt.figure(); plt.plot([r for r,_,_ in rows], [0 if e or o else 1 for r,e,o in rows], marker='o'); plt.ylim(-.1,1.1); plt.title('Large-rank nonbruteforce gate')
print('PASS', {'rows':rows,'enumeration_used':False})
